# A Compression Perspective on Simplicity Bias

This notebook implements the main experimental protocol from the paper.  For each training dataset size $N$ we:

1. **Experiment 1 — PCL curves**: train a family of feature-specific models (colour-based, digit-based, watermark-based) and estimate their prequential code length (PCL), which approximates $L_{\mathcal{L}}(p_N)$ — the description length of the learned model.
2. **Experiment 2 — Feature reliance**: train Neural Networks on the original mixed-feature dataset and measure which feature they actually rely at every each training dataset size $N$,  via accuracy on feature-specific test sets and permutation importance tests.

The key prediction: the feature selected by the MDL-optimal compressor (lowest total description length $L_{\mathcal{L}}(p) + N \cdot H(p^*, p)$) matches the feature the trained neural network relies on.


## Dataset Configurations

In [ ]:
# Imports and path setup
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import random
import os
import time
import importlib
import gc

from source import datasets
from source import algorithms
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm
from source.utils import hparams_registry
from source.datasets import CMNIST_TRAINING_DATASET_SIZES

from source.utils.notebook_helpers import (
    job_log,
    compute_pcl_curve,
    compute_bayes_optimal_pcl_curve,
    evaluate_accuracy,
    get_mean_log_loss_and_accuracy,
    compute_permutation_pvalue,
    _compute_kp_with_convergence_cutoff,
)

# Select device: use GPU if available
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    try:
        _ = torch.tensor(0.0, device=DEVICE)
        torch.cuda.synchronize()
    except Exception as e:
        job_log(f"WARNING: CUDA unavailable ({e}). Falling back to CPU.")
        DEVICE = torch.device('cpu')
print(f'Using device: {DEVICE}')

# Reproducibility
seed = int(os.environ.get('SEED', 12))
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(seed)
print(f'Using seed: {seed}')

# Import utility functions from misc.py
from source.utils.misc import (
    InfiniteDataLoader,
    get_base_dataset,
    attach_dataset_attributes,
    denormalize_cmnist,
 )

## Experiment settings

| Setting | Scenario | Features | Default `spur_prob` |
|---------|----------|----------|---------------------|
| 1 | **A** — spurious colour vs. robust digit | colour shortcut + digit | 0.1 (imbalanced envs) |
| 2 | **B** — robust digit vs. complex watermark | digit + env-specific watermark | 0.5 (balanced envs) |

Select the setting via the `EXPERIMENT_SETTING` environment variable (see `source/scripts/config.sh`).


In [ ]:
# ============================================================================
# EXPERIMENT SETTING SELECTOR
# ============================================================================
# Choose which experimental setting to use:
#   1 = Unbalanced groups (spur_prob=0.1, majority/minority) WITHOUT watermark
#       PCL curves: Color (majority only), Digit, Bayes
#   2 = Balanced groups (spur_prob=0.5) WITH watermark
#       PCL curves: Digit, Bayes
# ============================================================================

# Read from environment variable if set, otherwise use default
EXPERIMENT_SETTING = int(os.environ.get('EXPERIMENT_SETTING', '1'))

# Validate setting
assert EXPERIMENT_SETTING in [1, 2], f"EXPERIMENT_SETTING must be 1 or 2, got {EXPERIMENT_SETTING}"
print(f"Selected EXPERIMENT_SETTING = {EXPERIMENT_SETTING}")


In [ ]:
# Experiment config
DATASET_NAME = 'CMNIST'
DATA_PATH = os.environ.get('DATA_PATH', str(Path('data/benchmark').resolve()))
LEARNER = 'ERM'  # Simple SGD with regularization
BATCH_SIZE = int(os.environ.get('BATCH_SIZE', '64'))
NUM_FIT = 100  # number of points in the curve (dataset sizes)
NUM_WORKERS = 0

NETWORK_NAME = os.environ.get('NETWORK_NAME', 'MLP')
N_OUTPUTS = int(os.environ.get('N_OUTPUTS', '128'))
WEIGHT_DECAY = float(os.environ.get('WEIGHT_DECAY', '1e-4'))
OPTIMIZER_NAME = os.environ.get('OPTIMIZER_NAME', 'adamw')  # Options: 'sgd', 'adam', 'adamw'

#==============================================================================
DEBUG_MODE = os.environ.get('DEBUG_MODE', 'True').lower() in {'1', 'true', 'yes'}
#==============================================================================


DEBUG_MAX_STEPS = int(os.environ.get('DEBUG_MAX_STEPS', '5'))
DEBUG_DATASET_LIMIT = max(0, int(os.environ.get('DEBUG_DATASET_LIMIT', '1000')))

# Multiple runs configuration for small datasets
# For dataset sizes <= SMALL_DATA_THRESHOLD, run NUM_RUNS_SMALL times and average
SMALL_DATA_THRESHOLD = 200  # Dataset sizes <= this will use multiple runs
NUM_RUNS_SMALL = 10
NUM_RUNS_LARGE = 5
NUM_RUNS_MAX_SIZE = 10
N_PERMUTATIONS = 200
PERMUTATION_TEST_SAMPLES = 200
NATS_TO_BITS = 1.0 / np.log(2)

# Dataset complexity control
CMNIST_DIGITS_PER_CLASS = int(os.environ.get('CMNIST_DIGITS_PER_CLASS', '5'))
ESTIMATED_MAX_SIZE = int(CMNIST_TRAINING_DATASET_SIZES * (CMNIST_DIGITS_PER_CLASS / 5.0)) # Adjust max dataset size based on number of digits considered

dataset_sizes = list(set(np.logspace(np.log10(1), np.log10(ESTIMATED_MAX_SIZE), num=NUM_FIT, dtype=int)))

dataset_sizes.sort()
# Early stopping configuration
ES_MIN_DELTA = 5e-4
ES_PATIENCE = 5

if DEBUG_MODE:
    dataset_sizes = [1, 10, 100, 1000] 

    NUM_RUNS_SMALL = 2
    NUM_RUNS_LARGE = 2
    NUM_RUNS_MAX_SIZE = 5
    ES_PATIENCE = 1
    N_PERMUTATIONS = 10
    PERMUTATION_TEST_SAMPLES = 100
    DEBUG_DATASET_LIMIT = min(DEBUG_DATASET_LIMIT or 1000, 1000)
    print(f'DEBUG_MODE enabled: restricting dataset sizes to {dataset_sizes} with max {DEBUG_MAX_STEPS} training steps.')

RESULTS_BASE_DIR = Path(os.environ.get('RESULTS_BASE_DIR', '../results')).resolve()
RESULTS_BASE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_TIMESTAMP = os.environ.get('RESULT_FOLDER', time.strftime('%Y%m%d-%H%M%S'))
RESULTS_DIR = (RESULTS_BASE_DIR / RESULTS_TIMESTAMP).resolve()
if not DEBUG_MODE:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Config:')
print(f'  Dataset: {DATASET_NAME}')
print(f'  Learner: {LEARNER}')
print(f'  Optimizer: {OPTIMIZER_NAME}')
print(f'  Network: {NETWORK_NAME}')
print(f'  N_outputs: {N_OUTPUTS}')
print(f'  Data path: {DATA_PATH}')
print(f'  Batch size: {BATCH_SIZE}')
print(f'  Num workers: {NUM_WORKERS}')
print(f'  Digits per class: {CMNIST_DIGITS_PER_CLASS}')
print(f'  Estimated max size: {ESTIMATED_MAX_SIZE}')
print(f'  Dataset sizes: {dataset_sizes}')
print(f'  Weight decay: {WEIGHT_DECAY}')
print(f'  Small data threshold: {SMALL_DATA_THRESHOLD}')
print(f'  Runs for small datasets: {NUM_RUNS_SMALL}')
print(f'  Runs for large datasets: {NUM_RUNS_LARGE}')
print(f'  Runs at max size: {NUM_RUNS_MAX_SIZE}')
print(f'  Early stopping patience: {ES_PATIENCE}')
print(f'  Permutation test samples: {N_PERMUTATIONS}')
print(f'  Debug mode: {DEBUG_MODE}')
if DEBUG_MODE:
    print(f'  Debug dataset limit: {DEBUG_DATASET_LIMIT}')
    print(f'  Debug max steps: {DEBUG_MAX_STEPS}')
print(f'  Results directory: {RESULTS_DIR}')

## Hyperparameters

Initialise the network architecture and dataset-specific parameters.  All values are read from environment variables so that the notebook can be driven from `source/scripts/config.sh` without editing.


In [ ]:
from source.utils.misc import safe_float_env, safe_int_env

# Get base hyperparameters
hparams = hparams_registry.default_hparams(LEARNER, DATASET_NAME)

# Configure network architecture based on NETWORK_NAME
if NETWORK_NAME == 'MLP':
    hparams.update({
        'image_arch': 'simple_mlp',
        'input_size': 32,
        'lr': 1e-3,
        'n_outputs': N_OUTPUTS,
    })
    print('Using MLP architecture')
elif NETWORK_NAME == 'CNN':
    hparams.update({
        'image_arch': 'cnn',
        'input_size': 32,
        'lr': 1e-3,
        'n_outputs': N_OUTPUTS,
    })
    print('Using CNN architecture')
else:
    raise ValueError(f"Unknown network type: {NETWORK_NAME}. Choose 'MLP' or 'CNN'.")

# Override optimizer (default from hparams_registry is 'sgd' for images)
hparams['optimizer'] = OPTIMIZER_NAME
print(f'Using optimizer: {OPTIMIZER_NAME}')

# ============================================================================
# SETTING-BASED HYPERPARAMETERS (can be overridden via env vars)
# ============================================================================

if EXPERIMENT_SETTING == 1:
    # Setting 1: Unbalanced groups (spur_prob=0.1) WITHOUT watermark
    _default_spur_prob = 0.1
    has_watermark = False
    
elif EXPERIMENT_SETTING == 2:
    # Setting 2: Balanced groups (spur_prob=0.5) WITH watermark
    _default_spur_prob = 0.5
    has_watermark = True

# CMNIST-specific parameters
attr_prob = safe_float_env("ATTR_PROB", 0.5)
spur_prob = safe_float_env("SPUR_PROB", _default_spur_prob)
flip_prob = safe_float_env("FLIP_PROB", 0.09)
env_noisiness = safe_float_env("ENV_NOISINESS", 0.05)

watermark_bank_size = safe_int_env('WATERMARK_BANK_SIZE', 2)
watermark_bits = safe_int_env('WATERMARK_BITS', 32)

# Check for UNINFORMATIVE_MAJORITY (Setting 1 specific)
uninformative_majority = os.environ.get('UNINFORMATIVE_MAJORITY', 'False').lower() in {'1', 'true', 'yes'}

# Check for LOAD_PRETRAINED
load_pretrained = os.environ.get('LOAD_PRETRAINED', 'False').lower() in {'1', 'true', 'yes'}

hparams.update({
    'attr_prob': attr_prob,
    'spur_prob': spur_prob,
    'flip_prob': flip_prob,
    'env_noisiness': env_noisiness,
    'digits_per_class': CMNIST_DIGITS_PER_CLASS,
    'has_watermark': has_watermark,
    'watermark_bank_size': watermark_bank_size,
    'watermark_bits': watermark_bits,
    'uninformative_majority': uninformative_majority,
    'load_pretrained': load_pretrained,
    'weight_decay': WEIGHT_DECAY,
    'debug_mode': DEBUG_MODE,
    'debug_dataset_limit': DEBUG_DATASET_LIMIT
})

# ============================================================================
# EXPERIMENT METADATA - All hyperparameters for CSV storage
# This dictionary contains ALL parameters modifiable in run_multiple_notebook.sh
# to enable merging results from different experiments
# ============================================================================
EXPERIMENT_METADATA = {
    # Core experiment identifiers
    'experiment_setting': EXPERIMENT_SETTING,
    'seed': seed,
    'network_name': NETWORK_NAME,
    'learner': LEARNER,
    'dataset_name': DATASET_NAME,
    
    # Network hyperparameters
    'n_outputs': N_OUTPUTS,
    'optimizer': OPTIMIZER_NAME,
    'lr': hparams['lr'],
    'weight_decay': WEIGHT_DECAY,
    'load_pretrained': load_pretrained,
    
    # Dataset hyperparameters
    'attr_prob': attr_prob,
    'spur_prob': spur_prob,
    'flip_prob': flip_prob,
    'env_noisiness': env_noisiness,
    'digits_per_class': CMNIST_DIGITS_PER_CLASS,
    'watermark_bits': watermark_bits,
    
    # Experiment-specific parameters
    'has_watermark': has_watermark,
    'uninformative_majority': uninformative_majority,  # Setting 1 specific
    'watermark_bank_size': watermark_bank_size,    # Setting 2 specific
    
    # Training configuration (needed to reproduce aggregation)
    'batch_size': BATCH_SIZE,
    'num_runs_small': NUM_RUNS_SMALL,
    'num_runs_large': NUM_RUNS_LARGE,
    'small_data_threshold': SMALL_DATA_THRESHOLD,
    'es_patience': ES_PATIENCE,
    'es_min_delta': ES_MIN_DELTA,
    
    # Permutation test configuration
    'n_permutations': N_PERMUTATIONS,
    'permutation_test_samples': PERMUTATION_TEST_SAMPLES,
    
    # Debug settings
    'debug_mode': DEBUG_MODE,
    
    # Results path for traceability
    'results_folder': str(RESULTS_TIMESTAMP),
}

print(f'\nHyperparameters (Setting {EXPERIMENT_SETTING}):')
for key, val in hparams.items():
    print(f'  {key}: {val}')

print(f'\nExperiment Metadata (for CSV storage):')
for key, val in EXPERIMENT_METADATA.items():
    print(f'  {key}: {val}')

In [ ]:
# In debug mode, disable file writes to avoid creating artifacts
if DEBUG_MODE:
    import pandas as pd
    import matplotlib.pyplot as plt
    import os
    from pathlib import Path

    # Patch pandas DataFrame to_csv
    _pd_to_csv_orig = pd.DataFrame.to_csv
    def _pd_to_csv_noop(self, *args, **kwargs):
        target = args[0] if args else '<in-memory>'
        print(f"DEBUG_MODE: Skipping to_csv save to {target}")
        return None
    pd.DataFrame.to_csv = _pd_to_csv_noop

    # Patch matplotlib savefig
    _plt_savefig_orig = plt.savefig
    def _plt_savefig_noop(*args, **kwargs):
        target = args[0] if args else '<figure>'
        print(f"DEBUG_MODE: Skipping savefig to {target}")
        return None
    plt.savefig = _plt_savefig_noop

    # Patch os.makedirs
    _os_makedirs_orig = os.makedirs
    def _os_makedirs_noop(name, mode=0o777, exist_ok=False):
        print(f"DEBUG_MODE: Skipping makedirs for {name}")
        return None
    os.makedirs = _os_makedirs_noop

    # Patch Path.mkdir
    _path_mkdir_orig = Path.mkdir
    def _path_mkdir_noop(self, mode=0o777, parents=False, exist_ok=False):
        print(f"DEBUG_MODE: Skipping Path.mkdir for {self}")
        return None
    Path.mkdir = _path_mkdir_noop

    print("DEBUG_MODE: Patched to_csv, savefig, makedirs, and Path.mkdir to no-op (no files/folders will be written).")
else:
    print("DEBUG_MODE disabled: file saves are enabled.")

In [ ]:
# Debug helper RNG for reproducible subsets
import gc

debug_subset_rng = np.random.default_rng(seed + 42)

importlib.reload(datasets)
DatasetClass = getattr(datasets, DATASET_NAME)

# --- Dataset Preparation for Experiment 2 (Accuracy) ---
# These datasets are kept in memory for Experiment 2

train_dataset = DatasetClass(DATA_PATH, 'tr', hparams) 
val_dataset = DatasetClass(DATA_PATH, 'va', hparams)
test_dataset = DatasetClass(DATA_PATH, 'te', hparams)

# Define PCL hparams (datasets will be created one at a time in Experiment 1)
# Each PCL curve isolates ONE feature→label mapping by randomising the
# features that are NOT part of the chain being measured, while preserving
# the same noise levels (flip_prob, env_noisiness, spur_prob) as the
# original distribution so that model calibration is consistent.

# Color-based PCL: color → label (env0 only, no digit info)
pcl_color_hparams = hparams.copy()
pcl_color_hparams.update({
    'random_digit': True,
    'has_watermark': False,
})

pcl_bayes_hparams = hparams.copy()

# Digit-based PCL: digit_image → label (same noise as original)
pcl_digit_hparams = hparams.copy()
if EXPERIMENT_SETTING == 1:
    pcl_digit_hparams.update({
        'spur_prob': 0.5,
        'has_watermark': False,
    })

if EXPERIMENT_SETTING == 2:
    pcl_digit_hparams.update({
        'has_watermark': True,
        'random_watermark': True,
    })


# PCL Watermark-only: watermark → env → color → label (no digit info)
pcl_watermark_hparams = hparams.copy()
pcl_watermark_hparams.update({
    'has_watermark': True,
    'random_digit': True,
})

# Test set 1: Grayscale test data (balanced, no color information)
grayscale_hparams = hparams.copy()
grayscale_hparams.update({
    'grayscale': True,
    'has_watermark': False,
})
test_grayscale_dataset = DatasetClass(DATA_PATH, 'te', grayscale_hparams)


# Test set 4: Digit-only (no color-label correlation) - for Setting 1 only
digit_only_hparams = hparams.copy()
digit_only_hparams.update({
    'spur_prob': 0.5,  # No color-label correlation
    'has_watermark': False,
})
test_digit_only_dataset = DatasetClass(DATA_PATH, 'te', digit_only_hparams)

# Test set 2: Majority group (same correlation as training) - for Setting 1 only
test_majority_dataset = None
if EXPERIMENT_SETTING == 1:
    majority_hparams = hparams.copy()
    majority_hparams.update({
        'spur_prob': 0.0,  # Only majority group
        'has_watermark': False,
    })
    test_majority_dataset = DatasetClass(DATA_PATH, 'te', majority_hparams)

# Test set 5: Color-only (perfect color-label correlation, random digit) - for Setting 1 only
test_color_only_dataset = None
if EXPERIMENT_SETTING == 1:
    color_only_hparams = hparams.copy()
    color_only_hparams.update({
        'spur_prob': 0.0,  # Perfect color-label correlation
        'random_digit': True,  # Shuffle digits
        'has_watermark': False,
    })
    test_color_only_dataset = DatasetClass(DATA_PATH, 'te', color_only_hparams)

# Test set 3: Watermark only (noise digit + watermark + color) - for Setting 2
test_watermark_only_dataset = None
if EXPERIMENT_SETTING == 2:
    watermark_only_hparams = hparams.copy()
    watermark_only_hparams.update({
        'has_watermark': True,
        'random_digit': True,
    })
    test_watermark_only_dataset = DatasetClass(DATA_PATH, 'te', watermark_only_hparams)

pcl_digit_dataset = DatasetClass(DATA_PATH, 'tr', pcl_digit_hparams)


___
# Experiment 1: Prequential Code Length (PCL) curves

For each feature type (colour, digit, watermark) we train a *feature-specific* model — i.e. a model where all other features are shuffled — at a range of dataset sizes $N$.  The prequential code length approximates the total two-part description length:

$$\mathcal{J}(p, \mathcal{D}_N) = \underbrace{L_{\mathcal{L}}(p)}_{\text{model cost}} + N \cdot \underbrace{\mathbb{E}[-\log p(y|x)]}_{\text{data cost}}$$

**Interpreting $L_{\mathcal{L}}(p)$ (the model cost):**  
$L_{\mathcal{L}}(p)$ can be approximated as the area under the PCL curve minus the asymptotic prediction loss on held-out data (i.e. the average loss on the next unseen point).  Intuitively, the PCL curve starts high, when few examples have been seen, and decays toward the asymptotic loss $\ell_\infty$ as $N$ grows.  The excess area above $\ell_\infty$ captures how many bits were used to describe the pattern learned, which is the proxy for the model complexity cost $L_{\mathcal{L}}(p)$ we use.

The MDL-optimal feature at each $N$ is the one whose compression line lies lowest (the lower envelope).  Predicted transition points (vertical lines in the plots) mark where the envelope switches between feature types.

### Dataset Sample Visualization

Visual inspection of dataset samples after creation to verify correctness of data generation.

In [ ]:
# ============================================================================
# DEBUG VISUALIZATION: Sample images from created datasets
# ============================================================================
import matplotlib.pyplot as plt

def visualize_dataset_samples(dataset, title, n_samples=8, figsize=(12, 3)):
    """Visualize sample images from a dataset for debugging."""
    fig, axes = plt.subplots(1, n_samples, figsize=figsize)
    fig.suptitle(title, fontsize=14, fontweight='bold')
    
    for i in range(min(n_samples, len(dataset))):
        idx, img, label, attr, env, digit = dataset[i]
        
        # Denormalize and convert to numpy for plotting
        img_denorm = denormalize_cmnist(img).permute(1, 2, 0).numpy()
        img_denorm = np.clip(img_denorm, 0, 1)
        
        axes[i].imshow(img_denorm)
        axes[i].set_title(f"$y$={label.item()}\n$e$={env.item()}", fontsize=8)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

print("="*60)
print("DATASET SAMPLE VISUALIZATION FOR DEBUGGING")
print("="*60)

# Visualize training dataset
print(f"\n1. Training Dataset (n={len(train_dataset)}):")
print(f"   Setting: {EXPERIMENT_SETTING}, has_watermark={has_watermark}")
print(f"   spur_prob={spur_prob}, flip_prob={flip_prob}, env_noisiness={env_noisiness}")
visualize_dataset_samples(train_dataset, f"Training Dataset Samples (Setting {EXPERIMENT_SETTING})", n_samples=8)

# Visualize validation dataset
print(f"\n2. Validation Dataset (n={len(val_dataset)}):")
visualize_dataset_samples(val_dataset, "Validation Dataset Samples", n_samples=8)

# Visualize test dataset
print(f"\n3. Test Dataset (n={len(test_dataset)}):")
visualize_dataset_samples(test_dataset, "Test Dataset Samples", n_samples=8)

# Visualize no-watermark/grayscale test dataset
if EXPERIMENT_SETTING == 1:
    print(f"\n4. Grayscale Test Dataset (n={len(test_grayscale_dataset)}):")
    visualize_dataset_samples(test_grayscale_dataset, "Grayscale Test Dataset Samples", n_samples=8)
else:
    print(f"\n4. No-watermark Test Dataset (n={len(test_grayscale_dataset)}):")
    visualize_dataset_samples(test_grayscale_dataset, "No-watermark Test Dataset Samples", n_samples=8)

# Visualize majority test dataset (Setting 1 only)
if test_majority_dataset is not None:
    print(f"\n5. Majority Group Test Dataset (n={len(test_majority_dataset)}):")
    visualize_dataset_samples(test_majority_dataset, "Majority Group Test Dataset Samples", n_samples=8)

# Visualize watermark-only test dataset (Setting 2 only)
if test_watermark_only_dataset is not None:
    print(f"\n6. Watermark-only Test Dataset (n={len(test_watermark_only_dataset)}):")
    print("   (Should have random digits but correct watermarks)")
    visualize_dataset_samples(test_watermark_only_dataset, "Watermark-only Test Dataset Samples", n_samples=8)

# Visualize PCL digit dataset
print(f"\n7. PCL Digit Dataset (n={len(pcl_digit_dataset)}):")
print(f"   Used for computing digit-based PCL curves")
visualize_dataset_samples(pcl_digit_dataset, "PCL Digit Dataset Samples", n_samples=8)

In [ ]:
import pandas as pd
import math
import itertools

# Utility functions are imported from source.utils.notebook_helpers.
PCL_CONFIG = {
    'seed': seed,
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'small_data_threshold': SMALL_DATA_THRESHOLD,
    'num_runs_small': NUM_RUNS_SMALL,
    'num_runs_large': NUM_RUNS_LARGE,
    'num_runs_max_size': NUM_RUNS_MAX_SIZE,
    'es_min_delta': ES_MIN_DELTA,
    'es_patience': ES_PATIENCE,
    'debug_mode': DEBUG_MODE,
    'debug_max_steps': DEBUG_MAX_STEPS,
    'learner': LEARNER,
}

In [ ]:
# =============================================================================
# EXPERIMENT 1: PCL Computation (Memory-Efficient)
# =============================================================================

# Define PCL model configurations based on EXPERIMENT_SETTING
# =============================================================================
# Setting 1: Color (majority), Digit, Bayes
# Setting 2: Digit, Bayes (no color-based since groups are balanced)

if EXPERIMENT_SETTING == 1:
    attr_spec = ['color'] if spur_prob == 0 else ['color', 'digit']
    PERMUTATION_ATTRIBUTES_EXP1 = ['color', 'digit']
    pcl_configs = [
        {'name': 'Bayes-Optimal predictor', 'hparams': pcl_bayes_hparams, 'base_seed': seed + 202, 'attr_spec': attr_spec},
        {'name': 'Color-based', 'hparams': pcl_color_hparams, 'base_seed': seed + 101},
        {'name': 'Digit-based', 'hparams': pcl_digit_hparams, 'base_seed': seed + 303},
    ]
    print("PCL configs for Setting 1: Color-based, Digit-based, Bayes-Optimal")
elif EXPERIMENT_SETTING == 2:
    attr_spec = ['environment', 'color'] if env_noisiness == 0 else ['environment', 'color', 'digit']
    PERMUTATION_ATTRIBUTES_EXP1 = ['digit', 'watermark']
    pcl_configs = [
        {'name': 'Bayes-Optimal predictor', 'hparams': pcl_bayes_hparams, 'base_seed': seed + 202, 'attr_spec': attr_spec},
        {'name': 'Watermark-only', 'hparams': pcl_watermark_hparams, 'base_seed': seed + 404},
        {'name': 'Digit-based', 'hparams': pcl_digit_hparams, 'base_seed': seed + 303},
    ]
    print("PCL configs for Setting 2: Watermark-only, Digit-based, Bayes-Optimal")

PCL_MODEL_NAMES = [cfg['name'] for cfg in pcl_configs]
print(f"PCL models to compute: {PCL_MODEL_NAMES}")

# Build extra held-out evaluation datasets for all PCL models
# These allow measuring feature reliance (e.g. grayscale accuracy) at each dataset size
extra_eval_datasets = {
    'grayscale': test_grayscale_dataset,
}
if test_majority_dataset is not None:
    extra_eval_datasets['majority'] = test_majority_dataset
if test_watermark_only_dataset is not None:
    extra_eval_datasets['watermark_only'] = test_watermark_only_dataset
if test_digit_only_dataset is not None:
    extra_eval_datasets['digit_only'] = test_digit_only_dataset
if test_color_only_dataset is not None:
    extra_eval_datasets['color_only'] = test_color_only_dataset

print(f"Extra evaluation datasets: {list(extra_eval_datasets.keys())}")

# Create permutation test dataset (shared between Exp1 and Exp2)
resolved_test_dataset = get_base_dataset(test_dataset)
permutation_test_dataset = resolved_test_dataset.create_subset(
    n_samples=PERMUTATION_TEST_SAMPLES,
    seed=42
)

# Permutation test configuration for Experiment 1
exp1_permutation_config = {
    'dataset': permutation_test_dataset,
    'attributes': PERMUTATION_ATTRIBUTES_EXP1,
    'n_permutations': N_PERMUTATIONS,
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
}

print(f"Permutation test: {PERMUTATION_ATTRIBUTES_EXP1}, {N_PERMUTATIONS} permutations on {PERMUTATION_TEST_SAMPLES} samples")

pcl_results_list = []

for config in pcl_configs:
    model_name = config['name']
    model_hparams = config['hparams']
    model_seed = config['base_seed']
    
    print("\n" + "=" * 60)
    print(f"Processing: {model_name}")
    print("=" * 60)
    
    # --- Standard single-model PCL (non-Bayes) ---
    if 'attr_spec' not in config:
        print(f"Creating {model_name} datasets...")
        pcl_train_ds = DatasetClass(DATA_PATH, 'tr', model_hparams)
        pcl_val_ds = DatasetClass(DATA_PATH, 'va', model_hparams)
        pcl_test_ds = DatasetClass(DATA_PATH, 'te', model_hparams)
        
        print(f"  Train size: {len(pcl_train_ds)}, Val size: {len(pcl_val_ds)}, Test size: {len(pcl_test_ds)}")
        
        # 3. Compute PCL curve + permutation tests
        print(f"Computing PCL curve for {model_name}...")
        pcl_df = compute_pcl_curve(
            pcl_train_ds,
            pcl_val_ds,
            pcl_test_ds,  # Test distribution for test metrics
            test_dataset,  # Original test dataset for original loss
            model_hparams,
            DEVICE,
            dataset_sizes,
            base_seed=model_seed,
            config=PCL_CONFIG,
            job_logger=job_log,
            extra_eval_datasets=extra_eval_datasets,
            permutation_test_config=exp1_permutation_config,
            model_name=model_name,
        )
        pcl_df['model_type'] = model_name
        pcl_df['model_base_seed'] = model_seed
        pcl_results_list.append(pcl_df)
        
        del pcl_train_ds, pcl_val_ds
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        continue
    
    # --- Bayes-Optimal: per-size marginal log-likelihood + permutation tests ---
    attr_spec_list = [a.lower() for a in config['attr_spec']]
    bayes_df = compute_bayes_optimal_pcl_curve(
        DatasetClass, DATA_PATH, model_hparams, DEVICE, dataset_sizes, attr_spec_list,
        train_eval_dataset=train_dataset, test_eval_dataset=test_dataset,
        base_seed=model_seed, config=PCL_CONFIG, job_logger=job_log,
        extra_eval_datasets=extra_eval_datasets,
        permutation_test_config=exp1_permutation_config,
    )
    pcl_results_list.append(bayes_df)
    
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# Combine results
pcl_results_df = pd.concat(pcl_results_list, ignore_index=True)
for key, val in EXPERIMENT_METADATA.items():
    pcl_results_df[key] = val

pcl_csv_path = RESULTS_DIR / 'experiment1_pcl_results.csv'
pcl_results_df.to_csv(pcl_csv_path, index=False)

print("\n" + "=" * 60)
print("PCL computation complete! Results stored in DataFrame.")
print("=" * 60)

In [ ]:
# =============================================================================
# K(p) Calculation and Intersection Points (Robust to any PCL model list)
# =============================================================================

import numpy as np
import pandas as pd

# --- Calculate K(p) for each model available in pcl_results_df ---
k_p_results = {}

print("Calculating K(p) from PCL curves (using validation set)")
print(f"Available models: {pcl_results_df['model_type'].unique().tolist()}")

for model_type in pcl_results_df['model_type'].unique():
    # Ensure data is sorted by dataset_size
    model_df = pcl_results_df[pcl_results_df['model_type'] == model_type].sort_values('dataset_size')
    asymptotic_row = model_df.loc[model_df['dataset_size'].idxmax()]

    x = model_df['dataset_size'].values
    min_size = x[0]
    max_size = x[-1]
    width = max_size - min_size
    
    asymptotic_train_loss = float(asymptotic_row['mean_train_log_loss'])
    asymptotic_original_loss = float(asymptotic_row['mean_original_log_loss'])
    asymptotic_original_acc = float(asymptotic_row['mean_original_acc'])

    # Defaults: always defined so they are available regardless of which branch
    asymptotic_test_loss = float(asymptotic_row['mean_test_log_loss'])
    area_under_curve = np.nan
    
    # Determine asymptotic loss (height of the rectangle)
    if 'k_p_nats' in model_df.columns and not model_df['k_p_nats'].isna().all():
        k_p = float(model_df['k_p_nats'].iloc[-1])
        # Print per-subtask K(p) breakdown (stored by compute_bayes_optimal_pcl_curve)
        # Only consider columns that are not NaN for this model's rows to avoid
        # picking up columns from other models after pd.concat.
        subtask_cols = [
            c for c in model_df.columns
            if c.startswith('k_p_nats_') and model_df[c].notna().any()
        ]
        if subtask_cols:
            print(f"K(p) breakdown (total = sum of subtasks):")
            subtask_sum = 0.0
            for sc in subtask_cols:
                sc_val = float(model_df[sc].iloc[-1])
                subtask_sum += sc_val
                print(f"    {sc.replace('k_p_nats_', '')}: {sc_val:.4f} nats ({sc_val/np.log(2):.4f} bits)")
            print(f"    Total (from sum): {subtask_sum:.4f} nats ({subtask_sum/np.log(2):.4f} bits)")

    else:
        # Use convergence-cutoff technique for ALL non-Bayes (and Bayes fallback) models
        y = model_df['mean_test_log_loss'].values  # Using test loss for next-point prediction
        k_p, conv_idx, area_under_curve, asymptotic_test_loss, width_cut = _compute_kp_with_convergence_cutoff(x, y)
    
    if DEBUG_MODE:
        # Override K(p) and slopes for better visualization in debug mode
        if model_type == 'Color-based':
            k_p = 10 * np.log(2)  # 10 bits
            asymptotic_original_loss = 0.4 * np.log(2)
            print(f"DEBUG MODE: Overriding Color-based K(p)=10 bits, slope=0.4 bits/sample")
        elif model_type == 'Bayes-Optimal predictor':
            k_p = 100 * np.log(2)  # 100 bits
            asymptotic_original_loss = 0.2 * np.log(2)
            print(f"DEBUG MODE: Overriding Bayes-Optimal predictor K(p)=100 bits, slope=0.2 bits/sample")
        elif model_type == 'Digit-based':
            k_p = 50 * np.log(2)  # 50 bits
            asymptotic_original_loss = 0.3 * np.log(2)
            print(f"DEBUG MODE: Overriding Digit-based K(p)=50 bits, slope=0.3 bits/sample")

    
    k_p_results[model_type] = {
        # All values stored in NATS (convert to bits only at plotting time)
        'Kp_nats': k_p,
        'area_under_curve_nats': area_under_curve,
        'asymptotic_train_loss_nats': asymptotic_train_loss,
        'asymptotic_original_loss_nats': asymptotic_original_loss,
        'asymptotic_original_acc': asymptotic_original_acc,
        'asymptotic_test_loss_nats': float(asymptotic_row['mean_test_log_loss']),
        'asymptotic_test_acc': float(asymptotic_row['mean_test_acc']),
        'asymptotic_grayscale_acc': float(asymptotic_row['mean_grayscale_acc']),
        'asymptotic_grayscale_loss_nats': float(asymptotic_row['mean_grayscale_log_loss']),
        'asymptotic_watermark_only_acc': float(asymptotic_row.get('mean_watermark_only_acc', np.nan)) if 'mean_watermark_only_acc' in asymptotic_row.index else np.nan,
        'asymptotic_watermark_only_loss_nats': float(asymptotic_row.get('mean_watermark_only_log_loss', np.nan)) if 'mean_watermark_only_log_loss' in asymptotic_row.index else np.nan,

        'min_size': min_size,
        'max_size': max_size,
    }
    
    print(f"\n--- Model: {model_type} ---")
    print(f"  Dataset sizes range: {min_size} to {max_size}")

    print(f"  [Train] Asymptotic Mean Log Loss (slope, nats/sample): {asymptotic_train_loss:.4f}")
    print(f"  [Test] Asymptotic Mean Log Loss (slope, nats/sample): {asymptotic_test_loss:.4f}")
    print(f"  [Original] Asymptotic Mean Log Loss (slope, nats/sample): {asymptotic_original_loss :.4f}")
    print(f"  [Original] Asymptotic accuracy {asymptotic_original_acc:.4f}")
    print(f"  K(p) = {k_p:.4f} nats ({k_p / np.log(2):.4f} bits)")
    if not np.isnan(area_under_curve):
        print(f"  Total Area Under Curve (nats*examples): {area_under_curve:.2f}")
    print(f"  Estimated slope (bits/sample): {asymptotic_original_loss / np.log(2):.4f}")

# --- Calculate intersection points (robust to any model list) ---
print("\n" + "="*60)
print("Calculating MDL Curve Intersection Points")
print("="*60)

def find_intersection(k1, s1, name1, k2, s2, name2):
    """Calculate intersection point N = (k1 - k2) / (s2 - s1)"""
    if abs(s2 - s1) < 1e-9:
        print(f"Intersection between '{name1}' and '{name2}': Lines are parallel, no unique intersection.")
        return None
    
    intersection_N = (k1 - k2) / (s2 - s1)
    
    if intersection_N > 0:
        print(f"Intersection between '{name1}' and '{name2}': N = {intersection_N:.2f}")
        return intersection_N
    else:
        print(f"Intersection between '{name1}' and '{name2}': Occurs at N <= 0 (not in plotted range).")
        return None


# Convert from nats to bits here for intersection calculations
all_models = {}
for model_type, res in k_p_results.items():
    all_models[model_type] = {
        'k_bits': res['Kp_nats'] * NATS_TO_BITS,
        'slope_bits': res['asymptotic_original_loss_nats'] * NATS_TO_BITS
    }
    
# Always add Overfit model (K=0, slope=1 bit/sample)
k_overfit_bits = 0.0
slope_overfit_bits = -np.log(0.5) / np.log(2)  # 1.0 bits/sample

all_models['Overfit'] = {'k_bits': k_overfit_bits, 'slope_bits': slope_overfit_bits}

#print all_models
# Compute all pairwise intersections
intersections = {}
model_names = list(all_models.keys())
for i, name1 in enumerate(model_names):
    for name2 in model_names[i+1:]:
        k1, s1 = all_models[name1]['k_bits'], all_models[name1]['slope_bits']
        k2, s2 = all_models[name2]['k_bits'], all_models[name2]['slope_bits']
        key = f"{name1}_vs_{name2}".lower().replace(' ', '_').replace('-', '_')
        intersections[key] = find_intersection(k1, s1, name1, k2, s2, name2)


print(f"\nAll intersection points: {intersections}")

## Training Samples from the Proposed Dataset

Visualization of representative training samples from our semi-synthetic Watermarked CMNIST dataset.

In [ ]:
# Visualization of training samples from the proposed Watermarked CMNIST dataset
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

base_train_dataset = get_base_dataset(train_dataset)

# Sample 16 images (8 per row, 2 rows for images + 2 rows for watermarks)
n_samples = 16
sample_indices = list(range(n_samples))

fig = plt.figure(figsize=(10, 5))
# Use GridSpec for fine control: minimal space between image and watermark, more between pairs
gs = GridSpec(4, 8, figure=fig, height_ratios=[3, 1, 3, 1], wspace=0.05, hspace=0.10)

# Adjust spacing: reduce space between rows 0-1 and 2-3, keep some space between 1-2
axes = [[fig.add_subplot(gs[row, col]) for col in range(8)] for row in range(4)]
axes = np.array(axes)

# Add extra vertical space between the two image-watermark pairs
plt.subplots_adjust(hspace=0.02)
    
for row_pair in range(2):  # Two pairs of (image, watermark) rows
    img_row = row_pair * 2
    wm_row = row_pair * 2 + 1
    
    for col in range(8):
        sample_idx = row_pair * 8 + col
        i = sample_indices[sample_idx]
        idx, img, label, attr, env, digit = train_dataset[i]
        
        # Denormalize and convert to numpy for plotting
        img_denorm = denormalize_cmnist(img).permute(1, 2, 0).numpy()

        img_denorm = np.clip(img_denorm, 0, 1)

        # Image row
        axes[img_row, col].imshow(img_denorm)
        # Place title above the image with padding to avoid overlap with watermark
        axes[img_row, col].set_title(f"$y$={label.item()}, $e$={env.item()}", fontsize=10, pad=2)
        # Add thin black border
        for spine in axes[img_row, col].spines.values():
            spine.set_visible(True)
            spine.set_color('black')
            spine.set_linewidth(0.5)
        axes[img_row, col].set_xticks([])
        axes[img_row, col].set_yticks([])
        
        # Watermark row - extract directly from the denormalized image's rightmost column
        # This ensures the watermark displayed matches what's actually in the image
        rightmost_col = img_denorm[:, -1, :]  # Shape: (32, 3)
        # Convert to grayscale for binary display (take mean across channels)
        wm_from_img = rightmost_col.mean(axis=1)  # Shape: (32,)
        # Display vertically (as a column) to match the original image orientation
        # Use binary_r (reversed) so 1=black, 0=white for better visibility
        axes[wm_row, col].imshow(wm_from_img.reshape(-1, 1), cmap='binary_r', aspect='auto', vmin=0, vmax=1)
        # Add thin black border
        for spine in axes[wm_row, col].spines.values():
            spine.set_visible(True)
            spine.set_color('black')
            spine.set_linewidth(0.5)
        axes[wm_row, col].set_xticks([])
        axes[wm_row, col].set_yticks([])

plt.tight_layout()
#plt.savefig('training_samples.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
import math
import itertools
import pandas as pd

# Create validation loader (used for all models)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# Note: evaluate_accuracy and compute_permutation_pvalue are imported from notebook_helpers.

# Determine which attributes to test based on experiment setting
if EXPERIMENT_SETTING == 1:
    PERMUTATION_ATTRIBUTES = ['color', 'digit']
elif EXPERIMENT_SETTING == 2:
    # Setting 2 has watermarks, color is spurious
    PERMUTATION_ATTRIBUTES = ['digit', 'watermark']

print(f"Permutation test attributes: {PERMUTATION_ATTRIBUTES}")
print(f"Number of permutations: {N_PERMUTATIONS}")

# Reuse permutation_test_dataset from Experiment 1 (already created above)
# If not available (e.g. running Exp2 standalone), create it
if 'permutation_test_dataset' not in globals():
    resolved_test_dataset = get_base_dataset(test_dataset)
    permutation_test_dataset = resolved_test_dataset.create_subset(
        n_samples=PERMUTATION_TEST_SAMPLES, 
        seed=42
    )

train_dataset_size = len(train_dataset)
data_type = train_dataset.data_type
input_shape = train_dataset.INPUT_SHAPE
num_labels = train_dataset.num_labels
num_attributes = train_dataset.num_attributes

# === Main Training Loop ===
# This ensures same permutation is used across all dataset sizes for each run
all_results = []

# Determine how many runs to perform (use max to cover all sizes)
max_num_runs = max(NUM_RUNS_SMALL, NUM_RUNS_LARGE)

for run_idx in range(max_num_runs):
    print("\n" + "="*60)
    print(f"Run {run_idx + 1}/{max_num_runs}")
    job_log(f"[EXP2] Run index {run_idx + 1}/{max_num_runs}")
    print("="*60)
    
    # Set seed for this run (different seed for each run)
    run_seed = seed + run_idx * 1000
    torch.manual_seed(run_seed)
    np.random.seed(run_seed)
    random.seed(run_seed)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(run_seed)
    
    permuted_indices = np.random.permutation(train_dataset_size)
    
    # Now loop through dataset sizes with the SAME permutation
    for size_idx, size in enumerate(dataset_sizes):
        # Skip this run if exceeds required count for this dataset size
        run_count = NUM_RUNS_SMALL if size <= SMALL_DATA_THRESHOLD else NUM_RUNS_LARGE
        if run_idx >= run_count:
            continue
        print(f"\n  Dataset size: {size}")
        
        # Use the same permutation, just take first 'size' elements.
        # NOTE: Use Subset of the pre-built train_dataset (not a new CMNIST)
        # so that label/color/environment assignments are consistent with Exp1.
        # Creating a new CMNIST(..., subset_indices=...) would cause rng=666 to
        # operate on a shorter array, producing different per-sample assignments.
        subset_indices = permuted_indices[:size]
        train_subset = Subset(train_dataset, subset_indices.tolist())
        attach_dataset_attributes(train_subset)
        
        # Initialize algorithm
        AlgorithmClass = algorithms.get_algorithm_class(LEARNER)
        algo = AlgorithmClass(data_type, input_shape, num_labels, num_attributes, len(train_subset), hparams)
        algo = algo.to(DEVICE)
        
        if size_idx == 0:
            print(f"  Model initialized: {LEARNER} with {NETWORK_NAME} architecture")
        
        # Training setup
        checkpoint_freq = max(20, math.ceil(len(train_subset) / BATCH_SIZE))
        if DEBUG_MODE:
            checkpoint_freq = 1
        best_val_loss = float('inf')
        bad_checks = 0
        
        train_minibatches_iterator = iter(InfiniteDataLoader(
            dataset=train_subset,
            weights=None,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            seed=run_seed
        ))
        
        algo.train()
        pbar = tqdm(
            itertools.count(),
            desc=f"Training (run={run_idx+1}, size={size})",
            leave=False,
            disable=not DEBUG_MODE,
        )
        max_steps = DEBUG_MAX_STEPS if DEBUG_MODE else None
        
        for step in pbar:
            # Training step
            i, x, y, a, _, _ = next(train_minibatches_iterator)
            x, y, a = x.to(DEVICE), y.to(DEVICE), a.to(DEVICE)
            
            loss_dict = algo.update((i, x, y, a), step)
            
            # Validation checkpoint
            if (step + 1) % checkpoint_freq == 0:
                algo.eval()
                val_losses = []
                
                with torch.no_grad():
                    for i_val, x_val, y_val, a_val, e_val, d_val in val_loader:
                        x_val, y_val = x_val.to(DEVICE), y_val.to(DEVICE)
                        _, vloss = algo.predict(x_val, y_val, return_loss=True)
                        val_losses.append(vloss.item() if isinstance(vloss, torch.Tensor) else vloss)
                
                mean_val_loss = np.mean(val_losses)
                
                # Early stopping check
                if (best_val_loss - mean_val_loss) > ES_MIN_DELTA:
                    best_val_loss = mean_val_loss
                    bad_checks = 0
                else:
                    bad_checks += 1
                
                pbar.set_postfix({
                    'val_loss': f'{mean_val_loss:.4f}',
                    'best': f'{best_val_loss:.4f}',
                    'patience': f'{bad_checks}/{ES_PATIENCE}'
                })
                
                if bad_checks >= ES_PATIENCE and not DEBUG_MODE:
                    print(f"Early stopping at step {step+1}")
                    break
                
                algo.train()
            
            if DEBUG_MODE and max_steps is not None and (step + 1) >= max_steps:
                break
        
        pbar.close()
        
        # === Evaluation on validation and test distributions ===
        val_loss, val_acc = get_mean_log_loss_and_accuracy(algo, val_loader, DEVICE)
        print(f"    Validation loss: {val_loss:.4f} nats, accuracy: {val_acc:.4f}")
        
        # 1. Training data accuracy
        train_eval_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
        train_acc = evaluate_accuracy(algo, train_eval_loader, DEVICE)
        print(f"    Training data accuracy: {train_acc:.4f}")
        
        # 2. Grayscale test accuracy
        grayscale_loader = DataLoader(test_grayscale_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
        grayscale_acc = evaluate_accuracy(algo, grayscale_loader, DEVICE)
        print(f"    Grayscale test accuracy: {grayscale_acc:.4f}")
        
        # 3. Majority group test accuracy (Setting 1 only)
        majority_acc = np.nan
        if EXPERIMENT_SETTING == 1 and test_majority_dataset is not None:
            majority_loader = DataLoader(test_majority_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
            majority_acc = evaluate_accuracy(algo, majority_loader, DEVICE)
            print(f"    Majority group accuracy: {majority_acc:.4f}")

        # 4. Watermark-only test accuracy (Setting 2 only: noise digit + watermark)
        watermark_only_acc = np.nan
        if test_watermark_only_dataset is not None:
            watermark_only_loader = DataLoader(test_watermark_only_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
            watermark_only_acc = evaluate_accuracy(algo, watermark_only_loader, DEVICE)
            print(f"    Watermark-only accuracy: {watermark_only_acc:.4f}")

        # 5. Digit-only test accuracy (Setting 1 only: no color-label correlation)
        digit_only_acc = np.nan
        if test_digit_only_dataset is not None:
            digit_only_loader = DataLoader(test_digit_only_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
            digit_only_acc = evaluate_accuracy(algo, digit_only_loader, DEVICE)
            print(f"    Digit-only accuracy: {digit_only_acc:.4f}")

        # 6. Color-only test accuracy (Setting 1 only: perfect color, random digit)
        color_only_acc = np.nan
        if test_color_only_dataset is not None:
            color_only_loader = DataLoader(test_color_only_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
            color_only_acc = evaluate_accuracy(algo, color_only_loader, DEVICE)
            print(f"    Color-only accuracy: {color_only_acc:.4f}")

        # === Permutation Tests for Feature Reliance ===
        print(f"    Running permutation tests ({N_PERMUTATIONS} permutations on {PERMUTATION_TEST_SAMPLES} samples)...")
        pvalue_results = {}
        
        for attr in PERMUTATION_ATTRIBUTES:
            try:
                p_val, orig_acc, mean_shuf_acc = compute_permutation_pvalue(
                    algo, permutation_test_dataset, attr, DEVICE,
                    n_permutations=N_PERMUTATIONS,
                    batch_size=BATCH_SIZE,
                    num_workers=NUM_WORKERS,
                    base_seed=run_seed + 10000 + size_idx
                )
                pvalue_results[f'pvalue_{attr}'] = p_val
                pvalue_results[f'acc_drop_{attr}'] = orig_acc - mean_shuf_acc
                print(f"      {attr}: p-value={p_val:.3f}, acc_drop={orig_acc - mean_shuf_acc:+.4f}") 
            except ValueError as e:
                # Attribute not available (e.g., no watermark in setting 1)
                print(f"      {attr}: skipped ({e})")
                pvalue_results[f'pvalue_{attr}'] = np.nan
                pvalue_results[f'acc_drop_{attr}'] = np.nan
        
        result_entry = {
            'run_idx': run_idx,
            'dataset_size': size,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'train_acc': train_acc,
            'grayscale_acc': grayscale_acc,
            'majority_acc': majority_acc,
            'watermark_only_acc': watermark_only_acc,
            'digit_only_acc': digit_only_acc,
            'color_only_acc': color_only_acc,
        }
        # Add ALL experiment metadata to enable merging results from different experiments
        result_entry.update(EXPERIMENT_METADATA)
        result_entry['run_seed'] = run_seed  # Per-run seed (same for all sizes in this run)
        result_entry.update(pvalue_results)
        all_results.append(result_entry)
        
        del algo
        del train_eval_loader
        del train_minibatches_iterator
        del train_subset
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

# Create DataFrame from results
results_df = pd.DataFrame(all_results)
accuracy_csv_path = RESULTS_DIR / 'experiment2_accuracy_results.csv'
results_df.to_csv(accuracy_csv_path, index=False)

print("\n" + "="*60)
print("Training complete! Results stored in DataFrame.")
print("="*60)
print(results_df.head())
print(f"Saved Experiment 2 accuracy results to {accuracy_csv_path}")

___
# Results

The figure below shows, side by side for the chosen experiment setting:

- **Top row** — compression view: PCL loss curves, asymptotic compression lines $K(p) + N \cdot \ell_\infty$, and the full MDL envelope with intermediate models.
- **Bottom row** — learning view: accuracy on feature-specific test sets (solid = learned model, dashed = MDL prediction), permutation p-values, and accuracy drop.

Vertical purple lines mark the theoretical transition points predicted by the MDL framework.


In [ ]:
# =============================================================================
# Plotting: Aggregation + Envelope + MDL predictions
# =============================================================================
from source.utils.plotting import apply_publication_style, plot_experiment_summary

# ── Data source toggle ──────────────────────────────────────────────────
# Set LOAD_FROM_CSV = True and CSV_DIR to load data from saved CSVs
# instead of the current notebook session (useful for debugging plots).
LOAD_FROM_CSV = False
CSV_DIR = Path.cwd()  # e.g. Path('results/20260123-063253/setting2_.../0')

if LOAD_FROM_CSV:
    assert CSV_DIR is not None, 'Set CSV_DIR when LOAD_FROM_CSV is True'
    CSV_DIR = Path(CSV_DIR)
    pcl_results_df = pd.read_csv(CSV_DIR / 'experiment1_pcl_results.csv')
    results_df = pd.read_csv(CSV_DIR / 'experiment2_accuracy_results.csv')
    print(f'Loaded data from CSV: {CSV_DIR}')
    # Re-derive metadata from the loaded CSVs
    if 'experiment_setting' in pcl_results_df.columns:
        EXPERIMENT_SETTING = int(pcl_results_df['experiment_setting'].iloc[0])
    if 'spur_prob' in pcl_results_df.columns:
        spur_prob = float(pcl_results_df['spur_prob'].iloc[0])
    if 'flip_prob' in pcl_results_df.columns:
        flip_prob = float(pcl_results_df['flip_prob'].iloc[0])
    if 'env_noisiness' in pcl_results_df.columns:
        env_noisiness = float(pcl_results_df['env_noisiness'].iloc[0])
    if 'watermark_bank_size' in pcl_results_df.columns:
        watermark_bank_size = int(pcl_results_df['watermark_bank_size'].iloc[0])
    if 'watermark_bits' in pcl_results_df.columns:
        watermark_bits = int(pcl_results_df['watermark_bits'].iloc[0])
    if 'uninformative_majority' in pcl_results_df.columns:
        uninformative_majority = bool(pcl_results_df['uninformative_majority'].iloc[0])

# Apply publication-ready Matplotlib style
apply_publication_style()

# Use pre-computed K(p) from cell 18 when available; otherwise recompute from CSV
_kp = k_p_results if (not LOAD_FROM_CSV and 'k_p_results' in globals()) else None

combined_fig = plot_experiment_summary(
    pcl_results_df,
    results_df,
    k_p_results=_kp,
    experiment_setting=EXPERIMENT_SETTING,
    spur_prob=spur_prob,
    flip_prob=flip_prob,
    env_noisiness=env_noisiness,
    watermark_bank_size=watermark_bank_size,
    uninformative_majority=uninformative_majority,
    permutation_attributes=PERMUTATION_ATTRIBUTES,
    include_bayes_intermediates=False,
    show_non_envelope_lines=True,
    threshold_metric='k(p)',
    save_dir=RESULTS_DIR if not DEBUG_MODE else None,
    experiment_metadata=EXPERIMENT_METADATA,
)
plt.show()